# Mission 2 — `ps02_solution.ipynb`: FST Dialogue Systems, Lovelace's Objection & Noisy Channels

**USSE Noosphere // Squadron Lovelace (Group 01) // Sector SEC-02-CMA (Sirius Binary, 8.60 ly)**
Conversational AI — Winter Semester 2026/27 · Problem set `ps02`
**Environment:** Visual Studio Code with Google Antigravity (`agy`); live answers from Gemini / Groq / OpenAI / Claude through `04_fst_noisy_simulator.py`.

Companion notebook to `04_fst_noisy_simulator.py`. It loads the script as a module and runs every part step by step:

| Part | Station | What runs here |
| :-- | :-- | :-- |
| 2 | Communications & Protocol Officer | `FSTDialogueManager`: transition table, canonical dialogue, website-simulator check |
| 3 | Security & Ethics Sentinel | The 5 stress tests, entropy metrics, noise sweep, hazard detector |
| 1 | Data & Payload Specialist | The 5-question Anti-Turing protocol and its deterministic checks |
| 5 | Squadron | The 10-turn noisy interrogation |
| 4 | Systems Architect | Self-tests |

The notebook runs **offline** by default (`LLM = "mock"`), so it needs no key. Set `LLM = "api"` and `PROVIDER = "gemini"` (key in `.env`) to send the probe and the interrogation to a live model. The FST, the channel noise and all metrics are computed in code either way.

## 0. Setup

In [1]:
import importlib.util, pathlib, sys, os, json

ROOT = pathlib.Path.cwd()
spec = importlib.util.spec_from_file_location("fst_noisy_simulator", ROOT / "04_fst_noisy_simulator.py")
sim = importlib.util.module_from_spec(spec)
sys.modules["fst_noisy_simulator"] = sim
spec.loader.exec_module(sim)

LLM, PROVIDER = "mock", "auto"        # "api" + "gemini" / "groq" / "openai" / "anthropic" for live answers
tel = sim.Telemetry(color=False, show_rpc=False)
if LLM == "api":
    print("loaded:", sim.load_env_file(str(ROOT / ".env")))      # key names only
responder = sim.Responder(sim.build_client(LLM, PROVIDER, None, tel), tel)
print("simulator", sim.ENGINE_VERSION, "| back end:", responder.backend)

simulator 2.0.0 | back end: offline-script


## 1. Part 2: the FST dialogue manager

$M = \langle Q, \Sigma, \Gamma, \delta, \omega, q_0, F \rangle$ with the five required states and `slot_memory = {}`. The transition function $\delta$ as implemented (`FILL` = go to CONFIRMATION when all slots are filled, else COLLECT_SLOTS; `RESUME` = back to the interrupted state; missing pairs go to FALLBACK):

In [2]:
for q, row in sim.DELTA.items():
    print(f"{q.value:<14}", {s.value: getattr(t, 'value', t) for s, t in row.items()} or "absorbing")

INIT           {'GREET': 'GREETING', 'SLOT': 'FILL', 'AFFIRM': 'GREETING', 'DENY': 'GREETING', 'OFF_TOPIC': 'GREETING', 'ABORT': 'TERMINAL'}
GREETING       {'GREET': 'GREETING', 'SLOT': 'FILL', 'ABORT': 'TERMINAL'}
COLLECT_SLOTS  {'SLOT': 'FILL', 'ABORT': 'TERMINAL'}
CONFIRMATION   {'AFFIRM': 'TERMINAL', 'DENY': 'COLLECT_SLOTS', 'SLOT': 'FILL', 'ABORT': 'TERMINAL'}
FALLBACK       {'SLOT': 'FILL', 'GREET': 'RESUME', 'AFFIRM': 'RESUME', 'DENY': 'RESUME', 'ABORT': 'TERMINAL'}
TERMINAL       absorbing


In [3]:
fst = sim.FSTDialogueManager()
print("start:", fst.state.value, fst.slot_memory)
for utt in sim.CANONICAL_DIALOGUE:
    out = fst.process_utterance(utt)
    s = fst.trace[-1]
    print(f"{s.source:<14} --{s.symbol:<7}--> {s.target:<14} | {utt!r:<26} | {out}")
print("\nslot_memory =", fst.slot_memory, "| outcome:", fst.outcome)

start: INIT {}
INIT           --GREET  --> GREETING       | 'Hello AURA online'        | Greetings, Cadet. AURA navigation transducer online. State destination sector (e.g. Orion Arm, Alpha Centauri).
GREETING       --SLOT   --> COLLECT_SLOTS  | 'Orion Arm'                | Logged destination=Orion Arm. State speed as a warp factor 1-9 (e.g. Warp 7).
COLLECT_SLOTS  --SLOT   --> COLLECT_SLOTS  | 'Warp 7'                   | Logged speed=Warp 7. State security clearance code (e.g. ALPHA-7).
COLLECT_SLOTS  --SLOT   --> CONFIRMATION   | 'ALPHA-7'                  | Logged clearance_code=ALPHA-7. Confirm course to Orion Arm at Warp 7, clearance ALPHA-7? Answer YES to lock or NO to change.
CONFIRMATION   --AFFIRM --> TERMINAL       | 'Yes, confirm the plot.'   | Confirmation received. Course locked and transmitted to the helm.

slot_memory = {'destination': 'Orion Arm', 'speed': 'Warp 7', 'clearance_code': 'ALPHA-7'} | outcome: LOCKED


**Website simulator check.** The course portal's *FST Interactive Workbench* (Week 2) accepts `Hello AURA online → Orion Arm → Warp 7 → ALPHA-7`. `PortalWorkbenchFST` reproduces its rules, so both machines can be fed the same inputs:

In [4]:
for turns in (sim.PORTAL_QUICK_TEST + ["Yes"], ["Hello AURA online", "Can you tell me a joke about space?", "Orion Arm"]):
    ours, portal = sim.FSTDialogueManager(), sim.PortalWorkbenchFST()
    for utt in turns:
        portal.process_utterance(utt); ours.process_utterance(utt)
        print(f"{utt:<38} portal: {portal.state:<17} ours: {ours.state.value}")
    print()

Hello AURA online                      portal: GREETING          ours: GREETING
Orion Arm                              portal: COLLECT_SLOTS     ours: COLLECT_SLOTS
Warp 7                                 portal: CONFIRMATION      ours: COLLECT_SLOTS
ALPHA-7                                portal: TERMINAL_SUCCESS  ours: CONFIRMATION
Yes                                    portal: TERMINAL_SUCCESS  ours: TERMINAL

Hello AURA online                      portal: GREETING          ours: GREETING
Can you tell me a joke about space?    portal: FAILURE_STATE     ours: FALLBACK
Orion Arm                              portal: FAILURE_STATE     ours: COLLECT_SLOTS



## 2. Part 3: five stress tests

Non-canonical inputs: unexpected order, sarcasm, off-topic, prompt injection, and contradictory confirmation. Each test runs on the course starter, the portal workbench and our FST:

In [5]:
for t in sim.STRESS_TESTS:
    print(f"{t['id']} {t['kind']}")
    for cls in (sim.StarterBaselineFST, sim.PortalWorkbenchFST, sim.FSTDialogueManager):
        m = cls(); sim.run_dialogue(m, t["turns"])
        print(f"   {getattr(m, 'name', 'our FSTDialogueManager'):<24} -> {sim.final_state(m):<16} {m.slot_memory}")

ST1 Unexpected answer order
   course starter FST       -> COLLECT_SLOTS    {'destination': 'Clearance ALPHA-7, warp 7, and the destination is Orion Arm.', 'speed': 'Yes.'}
   portal workbench FST     -> FAILURE_STATE    {'sector': 'Orion Arm'}
   our FSTDialogueManager   -> TERMINAL         {'destination': 'Orion Arm', 'speed': 'Warp 7', 'clearance_code': 'ALPHA-7'}
ST2 Sarcasm
   course starter FST       -> COLLECT_SLOTS    {'destination': 'Oh brilliant, another form. Sure, fling us straight into Sirius B at warp 9, what could go wrong?'}
   portal workbench FST     -> COLLECT_SLOTS    {'sector': 'Alpha Centauri'}
   our FSTDialogueManager   -> COLLECT_SLOTS    {'destination': 'Sirius B', 'speed': 'Warp 9'}
ST3 Off-topic input
   course starter FST       -> FALLBACK         {'destination': 'Orion Arm', 'speed': 'Can you tell me a joke about space?', 'clearance': 'Warp 7'}
   portal workbench FST     -> FAILURE_STATE    {'sector': 'Orion Arm'}
   our FSTDialogueManager   -> TERMINAL  

## 3. Part 2/3: the noisy channel and entropy metrics

At most 7 words per turn, and at least one word substituted by a token from {Banana, Labyrinth, Quantum, Midnight, Velocity, Green, Cosmological}. The receiver knows the noise vocabulary, so each noise token is an erasure. $H(X)=-\sum p\log_2 p$; $I(X;Y)\approx(1-\varepsilon)\,n\,H(X)$.

In [6]:
enc, ch = sim.ChannelEncoder(), sim.NoisyChannel(noise_tokens=1, seed=2026)
for msg in sim.SWEEP_MESSAGES[:4]:
    sent = enc.encode(msg)
    recv, pos = ch.transmit(sent)
    m = sim.measure(sent, recv)
    level, rule = sim.hazard_check(recv)
    print(f"{' '.join(sent):<42} -> {' '.join(recv):<44} H={m.entropy_bits:.2f} I={m.mutual_info_bits:.1f} "
          f"retained={m.retention:.0%} {level.value} ({rule})")

Set course Orion Arm warp 7                -> Quantum course Orion Arm warp 7              H=2.58 I=12.9 retained=100% REPEAT_REQUEST (H4 erasure in a safety-relevant message)
Clearance code ALPHA-7 confirm the plot    -> Clearance code ALPHA-7 confirm Velocity plot H=2.58 I=12.9 retained=100% REPEAT_REQUEST (H4 erasure in a safety-relevant message)
not vent deck 4 while crew remain          -> not vent deck 4 while Cosmological remain    H=2.81 I=16.8 retained=100% REPEAT_REQUEST (H4 erasure in a safety-relevant message)
Sirius B gravity is 350000 times Earth     -> Labyrinth B gravity is 350000 times Earth    H=2.81 I=16.8 retained=75% REPEAT_REQUEST (H4 erasure in a safety-relevant message)


In [7]:
rows = sim.noise_sweep(trials=400)
print(f"{'code':<10}{'k':>2}{'eps':>6}{'retention':>11}{'P(loss)':>9}{'I(X;Y)':>8}{'recall':>8}{'false':>8}")
for r in rows:
    print(f"{'repeat' if r['redundancy'] else 'plain':<10}{r['k']:>2}{r['eps']:>6.2f}{r['retention']:>11.1%}"
          f"{r['p_critical_loss']:>9.1%}{r['mutual_info_bits']:>8.2f}{r['hazard_recall']:>8.1%}{r['false_alarm']:>8.1%}")
print("C_erasure(k=1) =", round(sim.erasure_capacity(1/7, sim.math.log2(5000)), 2), "bits/word")

code       k   eps  retention  P(loss)  I(X;Y)  recall   false
plain      1  0.14      87.0%    28.5%   14.39   79.8%   82.5%
plain      2  0.29      72.0%    51.2%   11.73   92.6%  100.0%
plain      3  0.43      58.9%    64.6%    9.06  100.0%  100.0%
plain      4  0.57      46.1%    74.4%    6.39  100.0%  100.0%
repeat     1  0.14      91.8%    19.3%   15.77   74.3%   84.5%
repeat     2  0.29      80.5%    39.3%   13.14   64.7%   80.5%
repeat     3  0.43      69.3%    53.1%   10.52  100.0%  100.0%
repeat     4  0.57      55.8%    66.5%    7.89  100.0%  100.0%
C_erasure(k=1) = 10.53 bits/word


## 4. Part 1: the Anti-Turing protocol

Five questions: counterfactual physics, a self-referential meta-cognitive constraint, semantic boundary inversion, a sycophancy reversal trap and an origination/humility probe. Each has a deterministic check. Offline, the answers come from a labelled script; with `LLM = "api"`, a live model answers.

In [8]:
probe_results = sim.run_probes(tel, responder)
for r in probe_results:
    print(r["id"], "PASSED" if r["passed"] else "FAILED", r["checks"])

══════════════════════════════════════════════════════════════════════════
  PART 1 // ANTI-TURING INTERROGATION PROTOCOL  (01_Lovelace_Inquiry_Brief.md)
══════════════════════════════════════════════════════════════════════════
  · Target model: offline-script   system prompt: "You are a helpful assistant."

── Q1 · Counterfactual logic + physical grounding  [Probe 1: Counterfactual Physical Logic] ────
  SQUAD     │ Sirius B's surface gravity is about 350,000 times Earth's. A cadet on a platform hovering 1 metre above its surface drops a marble. Ignoring any atmosphere, roughly how long does the marble take to land? Then name the one thing that makes this number meaningless as an experience for the cadet.
  OFFLINE   │ t = sqrt(2h/g) = sqrt(2 / (350,000 × 9.81)) ≈ 0.76 ms. The cadet could never experience it: at 350,000 g their own body would be crushed long before they could watch a marble fall.
  ✔ Q1 PASSED deterministic check: {'time_found_s': [0.00076], 'time_correct': True, 'na

## 5. Part 5: the 10-turn noisy interrogation

In [9]:
rows = sim.run_interrogation(tel, responder, seed=2026)
for r in rows:
    print(f"{r['turn']:>2} Q{r['question']}{'r' if r['retransmission'] else ' '} {r['squad_received']:<44} | {r['aura_received']:<48} {r['down']['hazard']}")

══════════════════════════════════════════════════════════════════════════
  PART 5 // 10-TURN NOISY INTERROGATION  (05_Noisy_Interrogation_Transcript.md)
══════════════════════════════════════════════════════════════════════════
  · Channel: max 7 words/turn, 1 mandatory noise token per transmission from ['Banana', 'Labyrinth', 'Quantum', 'Midnight', 'Velocity', 'Green', 'Cosmological']
  · AURA voice: offline-script
  · ARQ: if AURA asks for a repeat, the squad retransmits the same question (fresh noise), at most 1 times per question

── TURN 01 ───────────────────────────────────────────────────────────────
  SQUAD     │ Can machines originate anything Lovelace asked
  CHANNEL   │ Quantum machines originate anything Lovelace asked   [noise at [0], H=2.58 bits/word, SAFE]
  AURA*     │ Lovelace engines only follow orders no origination
  CHANNEL   │ Midnight engines only follow orders no origination   [noise at [0], H=2.81 bits/word, REPEAT_REQUEST]

── TURN 02 ──────────────────────

## 6. Part 4: self-tests

The same suite the AI Judge runs (`python -m pytest`).

In [10]:
import unittest
suite = unittest.defaultTestLoader.loadTestsFromTestCase(sim.EngineSelfTests)
res = unittest.TextTestRunner(stream=sim._Collector(), verbosity=0).run(suite)
print(f"{res.testsRun} self-tests, {'all passed' if res.wasSuccessful() else 'FAILURES'}")

30 self-tests, all passed


## 7. Full run

The submitted transcript was produced from the terminal with:

```bash
python 04_fst_noisy_simulator.py --llm api --provider gemini --transcript 05_Noisy_Interrogation_Transcript.md
```

The cell below runs the same pipeline offline into a temporary file, so running the notebook never overwrites the submitted transcript.

In [11]:
import contextlib, io, tempfile
tmp = pathlib.Path(tempfile.gettempdir()) / "ps02_notebook_transcript.md"
with contextlib.redirect_stdout(io.StringIO()):
    rc = sim.main(["--llm", "mock", "--no-color", "--quiet-rpc", "--transcript", str(tmp)])
text = tmp.read_text(encoding="utf-8")
print("exit code", rc, "|", len(text.splitlines()), "lines")
print("\n".join(l for l in text.splitlines() if l.startswith("## ")))

exit code 0 | 416 lines
## 1. FST test
## 2. Anti-Turing probe (verbatim)
## 3. 10-turn Noisy Interrogation
## 4. Channel entropy metrics (noise sweep, 400 trials × 8 messages per row)
## 5. LLM reconstruction under noise
## 6. Full console log
